In [1]:
# import packages
import time, math, os
from tqdm import tqdm
import gc
import pickle
import random
from datetime import datetime
from operator import itemgetter
import numpy as np
import pandas as pd
import warnings
import collections
warnings.filterwarnings('ignore')

In [2]:
# data_path = './data_raw/'
data_path = './tcdata/' # 天池平台路径
save_path = './temp_results/'  # 天池平台路径

In [10]:
metric_recall = True

In [3]:
from utils.memo_utils import reduce_mem
from utils.data_utils import get_all_click_sample, get_all_click_df
from utils.data_utils import get_user_item_time, get_item_info_df, get_item_info_dict

In [4]:
# 全量训练集
all_click_df = get_all_click_df(data_path, offline=False)
item_info_df = get_item_info_df(data_path)
item_type_dict, item_words_dict, item_created_time_dict = get_item_info_dict(item_info_df)

In [5]:
from models.CF import itemcf_sim, item_based_recommend

In [6]:
# i2i_sim = itemcf_sim(all_click_df, item_created_time_dict, save_path)

In [8]:
from utils.data_utils import get_hist_and_last_click, get_hist_and_last_click
from metrics import metrics_recall

In [11]:
# 先进行itemcf召回, 为了召回评估，所以提取最后一次点击
if metric_recall:
    trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df = all_click_df

user_recall_items_dict = collections.defaultdict(dict)
user_item_time_dict = get_user_item_time(trn_hist_click_df)

i2i_sim = pickle.load(open(save_path + 'itemcf_i2i_sim.pkl', 'rb'))
emb_i2i_sim = pickle.load(open(save_path + 'emb_i2i_sim.pkl', 'rb'))

sim_item_topk = 20
recall_item_num = 10
item_topk_click = get_item_topk_click(trn_hist_click_df, k=50)

for user in tqdm(trn_hist_click_df['user_id'].unique()):
    user_recall_items_dict[user] = item_based_recommend(user, user_item_time_dict, \
                                                        i2i_sim, sim_item_topk, recall_item_num, \
                                                        item_topk_click, item_created_time_dict, emb_i2i_sim)

user_multi_recall_dict['itemcf_sim_itemcf_recall'] = user_recall_items_dict
pickle.dump(user_multi_recall_dict['itemcf_sim_itemcf_recall'], open(save_path + 'itemcf_recall_dict.pkl', 'wb'))

if metric_recall:
    # 召回效果评估
    metrics_recall(user_multi_recall_dict['itemcf_sim_itemcf_recall'], trn_last_click_df, topk=recall_item_num)

KeyError: 'user_id'

In [ ]:
# 将字典的形式转换成df
user_item_score_list = []

for user, items in tqdm(user_recall_items_dict.items()):
    for item, score in items:
        user_item_score_list.append([user, item, score])

recall_df = pd.DataFrame(user_item_score_list, columns=['user_id', 'click_article_id', 'pred_score'])

In [ ]:
# 生成提交文件
from utils.submit_utils import submit

In [ ]:
# 获取测试集
tst_click = pd.read_csv(data_path + 'testA_click_log.csv')
tst_users = tst_click['user_id'].unique()

# 从所有的召回数据中将测试集中的用户选出来
tst_recall = recall_df[recall_df['user_id'].isin(tst_users)]

# 生成提交文件
submit(tst_recall, topk=5, model_name='itemcf_baseline')